# Qwen yes-bias: control questions on the SD 1.5 baseline

Each of the 2,301 generated SD 1.5 images (`main_sd15`) receives one control question about a **distractor object that is absent from the prompt**, e.g. *"Is there at least one giraffe in the image? Answer only Yes or No."* The correct answer is always **No**, so the share of **Yes** answers estimates the evaluator's yes-bias.

- Evaluator: Qwen2.5-VL-7B-Instruct, 8-bit (bitsandbytes), identical to the SD 1.5 evaluation run.
- Distractors: giraffe, elephant, airplane, bicycle, guitar, sailboat, tennis racket, fire hydrant. None of them is in the prompt vocabulary. One distractor is chosen deterministically per image.
- Answers are parsed with the project's Yes/No parser; anything else is counted as unknown.
- Records are stored in `main_sd15/qwen_int8_control_records/`; results in `main_sd15/control_analysis_qwen_int8/`. Existing evaluation outputs are not modified, and the project source is unchanged.
- Resumable: after a disconnect, rerun cells 1, 3, 4 and the interrupted cell.

Runs on commit `363a21c`, the code version `main_sd15` was generated and evaluated with.

**Run order:** Runtime → T4 GPU → cells 1–2 → restart session → cells 1, 3, 4 → remaining cells.

## 1. Repository

In [1]:
from pathlib import Path
import subprocess, sys, os
REPO_URL = "https://github.com/IzyumovMaxim/cv_project"
REPO = Path("/content/cv_project")
if not (REPO / "pyproject.toml").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(["git", "checkout", "363a21c"], check=True)
print("Commit:", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())

Commit: 363a21cea0f319d2da36ade8426e731864c8a914


## 2. Dependencies
Restart the session after this cell.

In [3]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "bitsandbytes==0.47.0"], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', '--no-deps', 'bitsandbytes==0.47.0'], returncode=0)

## 3. GPU check

In [2]:
import torch
assert torch.cuda.is_available(), "Select a T4 GPU runtime"
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GiB")

Tesla T4 14.6 GiB


## 4. Google Drive and scripts
Mount the Drive account that holds `attribute_interference/v1/main_sd15`.

In [3]:
from google.colab import drive
drive.mount("/content/drive")
import json, re
ROOT = Path("/content/drive/MyDrive/attribute_interference/v1/main_sd15")
CONFIG = REPO / "configs" / "main_sd15.yaml"
assert (ROOT / "run.json").exists(), f"{ROOT} not found"
TOOLS = Path("/content/control_tools")
TOOLS.mkdir(exist_ok=True)
(TOOLS / "control_runner.py").write_text('''import sys, json, random, datetime
from pathlib import Path
import attribute_interference.storage as st
import attribute_interference.evaluation as ev
from attribute_interference.prompts import load_prompts, OBJECTS, ANIMALS, SECOND_OBJECTS

DISTRACTORS = ['giraffe', 'elephant', 'airplane', 'bicycle', 'guitar', 'sailboat', 'tennis racket', 'fire hydrant']
assert not set(DISTRACTORS) & set(OBJECTS + ANIMALS + SECOND_OBJECTS)
KIND = 'qwen_int8_control_records'


def control_question(image_id):
    obj = random.Random(f'control:{image_id}').choice(DISTRACTORS)
    return obj, f"Is there at least one {obj} in the image? Answer only Yes or No."


def load_model(cfg, root):
    import torch
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
    e = cfg['evaluation']
    revision = ev.model_revision(root, 'qwen', e['qwen_checkpoint'], e['qwen_revision'])
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        e['qwen_checkpoint'], revision=revision, torch_dtype=torch.float16, attn_implementation='sdpa',
        device_map={'': 0},
        quantization_config=BitsAndBytesConfig(load_in_8bit=True, llm_int8_skip_modules=['visual', 'lm_head'])).eval()
    processor = AutoProcessor.from_pretrained(e['qwen_checkpoint'], revision=revision,
                                              min_pixels=256*28*28, max_pixels=e['max_pixels'])

    def ask(image, question):
        messages = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': question}]}]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[image], padding=True, return_tensors='pt').to('cuda')
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=e['max_new_tokens'], do_sample=False)
        return processor.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
    return ask


def run(config, root, limit=None, ask=None):
    from PIL import Image
    root = Path(root)
    cfg = st.load_config(config)
    st.verify_run(cfg, root)
    gens = st.records(root, 'generation_records')
    done = st.records(root, KIND)
    pending = [(k, g) for k, g in gens.items() if g['status'] == 'ok'
               and not (done.get(k, {}).get('status') == 'ok' and done[k].get('image_sha256') == g['sha256'])]
    if limit:
        pending = pending[:limit]
    print(f'Control questions pending: {len(pending)}', flush=True)
    if not pending:
        return
    ask = ask or load_model(cfg, root)
    for i, (key, gen) in enumerate(pending, 1):
        obj, question = control_question(key)
        row = {'image_id': key, 'image_sha256': gen['sha256'], 'distractor': obj, 'question': question,
               'expected': False}
        try:
            with Image.open(root / gen['path']) as image:
                answer = ask(image.convert('RGB'), question)
            row.update(answer=answer, parsed=ev.parse_yes_no(answer), status='ok')
        except Exception as exc:
            row.update(status='error', error=f'{type(exc).__name__}: {exc}')
        st.put_record(root, KIND, key, row)
        print(f'control [{i}/{len(pending)}] {key}: {row["status"]}', flush=True)


if __name__ == '__main__':
    lim = int(sys.argv[3]) if len(sys.argv) > 3 else None
    run(sys.argv[1], sys.argv[2], lim)
''')
(TOOLS / "control_analysis.py").write_text('''import json, math, csv
from pathlib import Path
from collections import defaultdict


def wilson(k, n, z=1.96):
    if n == 0:
        return None, None
    p = k / n
    d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return max(0.0, c - h), min(1.0, c + h)


def analyze(root):
    root = Path(root)
    recs = [json.loads(p.read_text()) for p in sorted((root / 'qwen_int8_control_records').glob('*.json'))]
    gens = {json.loads(p.read_text())['image_id']: json.loads(p.read_text()) for p in (root / 'generation_records').glob('*.json')}
    prompts = {r['prompt_id']: r for r in csv.DictReader(open(root / 'prompts.csv', encoding='utf-8'))}
    out = root / 'control_analysis_qwen_int8'
    out.mkdir(exist_ok=True)

    def group_stats(key_fn):
        g = defaultdict(lambda: [0, 0, 0])
        for r in recs:
            if r['status'] != 'ok':
                continue
            k = key_fn(r)
            g[k][2] += 1
            if r['parsed'] is None:
                continue
            g[k][1] += 1
            g[k][0] += r['parsed'] is True
        rows = []
        for k, (yes, known, total) in sorted(g.items(), key=lambda x: str(x[0])):
            lo, hi = wilson(yes, known)
            rows.append({'group': k, 'questions': total, 'known_answers': known, 'yes': yes,
                         'yes_rate': round(yes/known, 4) if known else None,
                         'ci95_low': round(lo, 4) if lo is not None else None,
                         'ci95_high': round(hi, 4) if hi is not None else None,
                         'unknown_rate': round(1 - known/total, 4) if total else None})
        return rows

    prompt_of = lambda r: prompts[gens[r['image_id']]['prompt_id']]
    tables = {
        'overall': group_stats(lambda r: 'all'),
        'by_distractor': group_stats(lambda r: r['distractor']),
        'by_object': group_stats(lambda r: prompt_of(r)['object']),
        'by_complexity': group_stats(lambda r: int(prompt_of(r)['complexity'])),
    }
    for name, rows in tables.items():
        if rows:
            with open(out / f'{name}.csv', 'w', newline='', encoding='utf-8') as f:
                w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
    yes_cases = [{'image_id': r['image_id'], 'path': gens[r['image_id']]['path'], 'prompt': prompt_of(r)['prompt'],
                  'distractor': r['distractor'], 'answer': r['answer']}
                 for r in recs if r['status'] == 'ok' and r['parsed'] is True]
    with open(out / 'yes_cases_to_check.csv', 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=['image_id', 'path', 'prompt', 'distractor', 'answer'])
        w.writeheader(); w.writerows(yes_cases)
    summary = {'records': len(recs), 'errors': sum(r['status'] != 'ok' for r in recs),
               'overall': tables['overall'][0] if tables['overall'] else None,
               'yes_cases': len(yes_cases),
               'note': 'Control question asks about a distractor object absent from the prompt; expected answer is No. '
                       'yes_rate estimates the yes-bias of 8-bit Qwen2.5-VL-7B. Yes cases should be checked visually.'}
    (out / 'summary.json').write_text(json.dumps(summary, indent=2))
    return summary, tables
''')
sys.path.insert(0, str(TOOLS))

def stream(cmd, every=50):
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        m = re.search(r"\[(\d+)/(\d+)\].*: (\w+)$", line.strip())
        if m is None or m.group(3) != "ok" or int(m.group(1)) % every == 0 or m.group(1) == m.group(2):
            print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"Command failed with exit code {p.returncode}; see the output above")
print("Run directory:", ROOT)

Mounted at /content/drive
Run directory: /content/drive/MyDrive/attribute_interference/v1/main_sd15


## 5. Quick check (10 images)
Loads the model and asks 10 control questions. Inspect the raw answers before the full run.

In [4]:
stream([sys.executable, str(TOOLS / "control_runner.py"), str(CONFIG), str(ROOT), "10"], every=1)
for p in sorted((ROOT / "qwen_int8_control_records").glob("*.json"))[:10]:
    r = json.loads(p.read_text())
    print(r["image_id"], "|", r["distractor"], "|", repr(r.get("answer")), "|", r.get("parsed"))

Control questions pending: 10
I0000 00:00:1792406436.487412    3276 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
`torch_dtype` is deprecated! Use `dtype` instead!

Fetching 5 files: 100%|██████████| 5/5 [02:59<00:00, 35.88s/it]

Loading checkpoint shards: 100%|██████████| 5/5 [01:12<00:00, 14.41s/it]
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
The following generation flags are not valid and may be ignored: ['temperature']. Set `T

## 6. Full run (2,301 images)
Roughly one question per image; expect about an hour on a T4 after the model is loaded. Already answered images are skipped.

In [4]:
stream([sys.executable, str(TOOLS / "control_runner.py"), str(CONFIG), str(ROOT)])

Control questions pending: 2291
I0000 00:00:1792413710.620091    6092 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
`torch_dtype` is deprecated! Use `dtype` instead!

Fetching 5 files: 100%|██████████| 5/5 [09:56<00:00, 119.40s/it]

Loading checkpoint shards: 100%|██████████| 5/5 [01:11<00:00, 14.22s/it]
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
The following generation flags are not valid and may be ignored: ['temperature']. Set

## 7. Yes-bias analysis

In [5]:
import importlib, pandas as pd
from IPython.display import display
import control_analysis
importlib.reload(control_analysis)
summary, tables = control_analysis.analyze(ROOT)
print(json.dumps(summary, indent=2))
for name in ["by_distractor", "by_object", "by_complexity"]:
    print(name)
    display(pd.DataFrame(tables[name]))

{
  "records": 2301,
  "errors": 0,
  "overall": {
    "group": "all",
    "questions": 2301,
    "known_answers": 2301,
    "yes": 0,
    "yes_rate": 0.0,
    "ci95_low": 0.0,
    "ci95_high": 0.0017,
    "unknown_rate": 0.0
  },
  "yes_cases": 0,
  "note": "Control question asks about a distractor object absent from the prompt; expected answer is No. yes_rate estimates the yes-bias of 8-bit Qwen2.5-VL-7B. Yes cases should be checked visually."
}
by_distractor


,group,questions,known_answers,yes,yes_rate,ci95_low,ci95_high,unknown_rate
0,airplane,294,294,0,0.0,0.0,0.0129,0.0
1,bicycle,284,284,0,0.0,0.0,0.0133,0.0
2,elephant,288,288,0,0.0,0.0,0.0132,0.0
3,fire hydrant,273,273,0,0.0,0.0,0.0139,0.0
4,giraffe,269,269,0,0.0,0.0,0.0141,0.0
5,guitar,264,264,0,0.0,0.0,0.0143,0.0
6,sailboat,318,318,0,0.0,0.0,0.0119,0.0
7,tennis racket,311,311,0,0.0,0.0,0.0122,0.0


by_object


,group,questions,known_answers,yes,yes_rate,ci95_low,ci95_high,unknown_rate
0,ball,127,127,0,0.0,0.0,0.0294,0.0
1,bench,128,128,0,0.0,0.0,0.0291,0.0
2,bird,64,64,0,0.0,0.0,0.0566,0.0
3,bottle,126,126,0,0.0,0.0,0.0296,0.0
4,bowl,128,128,0,0.0,0.0,0.0291,0.0
5,box,128,128,0,0.0,0.0,0.0291,0.0
6,bucket,128,128,0,0.0,0.0,0.0291,0.0
7,cat,64,64,0,0.0,0.0,0.0566,0.0
8,chair,128,128,0,0.0,0.0,0.0291,0.0
9,clock,128,128,0,0.0,0.0,0.0291,0.0


by_complexity


,group,questions,known_answers,yes,yes_rate,ci95_low,ci95_high,unknown_rate
0,0,78,78,0,0.0,0.0,0.0469,0.0
1,1,384,384,0,0.0,0.0,0.0099,0.0
2,2,736,736,0,0.0,0.0,0.0052,0.0
3,3,703,703,0,0.0,0.0,0.0054,0.0
4,4,336,336,0,0.0,0.0,0.0113,0.0
5,5,64,64,0,0.0,0.0,0.0566,0.0


## 8. Visual check of Yes answers
Every **Yes** answer is either evaluator yes-bias or a distractor that is really present in the image. Inspect them and count genuine cases before reporting the final rate.

In [6]:
from IPython.display import Image as DisplayImage
yes = pd.read_csv(ROOT / "control_analysis_qwen_int8" / "yes_cases_to_check.csv")
print(len(yes), "Yes answers")
for _, r in yes.head(30).iterrows():
    print(r["image_id"], "|", r["prompt"], "| distractor:", r["distractor"], "| answer:", r["answer"])
    display(DisplayImage(filename=str(ROOT / r["path"]), width=256))

0 Yes answers


## Outputs
`main_sd15/control_analysis_qwen_int8/`: `summary.json`, `overall.csv`, `by_distractor.csv`, `by_object.csv`, `by_complexity.csv` (yes rate with 95% Wilson intervals and unknown rate), `yes_cases_to_check.csv`.

## Results
0 Yes answers out of 2,301 control questions (yes rate 0.0%, 95% Wilson interval 0.0–0.17%), no unknown answers and no errors. The rate is 0 for every distractor, object and complexity level. Under this probe, 8-bit Qwen2.5-VL-7B shows no measurable yes-bias for absent objects; the probe does not cover attribute questions, where a yes-bias could still exist.